In [3]:
import pandas as pd
import numpy as np
url = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv'
df = pd.read_csv(url)

In [2]:
df[['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']].isnull().sum()

engine_displacement      0
horsepower             708
vehicle_weight           0
model_year               0
dtype: int64

In [3]:
df['horsepower'].median()

149.0

In [1]:
def prepare_X(df, fill_value):
    df_num = df[base]
    df_num = df_num.fillna(fill_value)
    X = df_num.values
    return X

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

# Case 1: Fill with 0
X_train_zero = prepare_X(df_train, 0)
w_0, w_zero = train_linear_regression(X_train_zero, y_train)
X_val_zero = prepare_X(df_val, 0)
y_pred_zero = w_0 + X_val_zero.dot(w_zero)
score_zero = round(rmse(y_val, y_pred_zero), 2)

# Case 2: Fill with mean
mean_hp = df_train['horsepower'].mean()
X_train_mean = prepare_X(df_train, mean_hp)
w_0, w_mean = train_linear_regression(X_train_mean, y_train)
X_val_mean = prepare_X(df_val, mean_hp)
y_pred_mean = w_0 + X_val_mean.dot(w_mean)
score_mean = round(rmse(y_val, y_pred_mean), 2)

print("Score with 0:", score_zero)
print("Score with mean:", score_mean)

NameError: name 'df_train' is not defined

In [4]:
# 1. Kuandaa Data na Kugawa (Train/Val/Test)
base = ['engine_displacement', 'num_cylinders', 'horsepower', 'vehicle_weight', 'acceleration', 'model_year']

n = len(df)
n_val = int(0.2 * n)
n_test = int(0.2 * n)
n_train = n - (n_val + n_test)

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_shuffled = df.iloc[idx]

df_train = df_shuffled.iloc[:n_train].copy()
df_val = df_shuffled.iloc[n_train:n_train+n_val].copy()
df_test = df_shuffled.iloc[n_train+n_val:].copy()

y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
y_val = np.log1p(df_val.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

# 2. Functions za Training na Evaluation
def prepare_X(df, fill_value):
    df_num = df[base]
    df_num = df_num.fillna(fill_value)
    X = df_num.values
    return X

def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

def rmse(y, y_pred):
    error = y_pred - y
    mse = (error ** 2).mean()
    return np.sqrt(mse)

# 3. Case 1: Fill with 0
X_train_zero = prepare_X(df_train, 0)
w_0, w_zero = train_linear_regression(X_train_zero, y_train)
X_val_zero = prepare_X(df_val, 0)
y_pred_zero = w_0 + X_val_zero.dot(w_zero)
score_zero = round(rmse(y_val, y_pred_zero), 2)

# 4. Case 2: Fill with mean
mean_hp = df_train['horsepower'].mean()
X_train_mean = prepare_X(df_train, mean_hp)
w_0, w_mean = train_linear_regression(X_train_mean, y_train)
X_val_mean = prepare_X(df_val, mean_hp)
y_pred_mean = w_0 + X_val_mean.dot(w_mean)
score_mean = round(rmse(y_val, y_pred_mean), 2)

print("Score with 0:", score_zero)
print("Score with mean:", score_mean)

Score with 0: 0.04
Score with mean: 0.04


In [5]:
def train_linear_regression_reg(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X)
    reg = r * np.eye(XTX.shape[0])
    XTX = XTX + reg
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv.dot(X.T).dot(y)
    return w[0], w[1:]

X_train_zero = prepare_X(df_train, 0)
X_val_zero = prepare_X(df_val, 0)

for r in [0, 0.01, 0.1, 1, 5, 10]:
    w_0, w = train_linear_regression_reg(X_train_zero, y_train, r=r)
    y_pred = w_0 + X_val_zero.dot(w)
    print(f"r={r:<5} RMSE={round(rmse(y_val, y_pred), 2)}")

r=0     RMSE=0.04
r=0.01  RMSE=0.04
r=0.1   RMSE=0.04
r=1     RMSE=0.04
r=5     RMSE=0.04
r=10    RMSE=0.04


In [6]:
scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_shuffled = df.iloc[idx]

    df_train = df_shuffled.iloc[:n_train].copy()
    df_val = df_shuffled.iloc[n_train:n_train+n_val].copy()

    y_train = np.log1p(df_train.fuel_efficiency_mpg.values)
    y_val = np.log1p(df_val.fuel_efficiency_mpg.values)

    X_train = prepare_X(df_train, 0)
    w_0, w = train_linear_regression(X_train, y_train)

    X_val = prepare_X(df_val, 0)
    y_pred = w_0 + X_val.dot(w)

    score = rmse(y_val, y_pred)
    scores.append(score)

std_score = np.std(scores)
print("Q5 Standard Deviation:", round(std_score, 3))

Q5 Standard Deviation: 0.001


In [7]:
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_shuffled = df.iloc[idx]

df_train = df_shuffled.iloc[:n_train].copy()
df_val = df_shuffled.iloc[n_train:n_train+n_val].copy()
df_test = df_shuffled.iloc[n_train+n_val:].copy()

df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

y_full_train = np.log1p(df_full_train.fuel_efficiency_mpg.values)
y_test = np.log1p(df_test.fuel_efficiency_mpg.values)

X_full_train = prepare_X(df_full_train, 0)
X_test = prepare_X(df_test, 0)

w_0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)
y_pred = w_0 + X_test.dot(w)

score = rmse(y_test, y_pred)
print("Q6 Test RMSE:", round(score, 2))

Q6 Test RMSE: 0.04
